In [ ]:
import os
module_dir = r"D:/ForestFire/CrownFire/src"
os.chdir(module_dir)
from CBH import nfiPreprocessing, func, loss_func, species_weighted_r2, objective, printFeatureImportance, \
hybrid_model, create_valid_mask, buildDistribution, visualize_multiple_distribution, GPUSamplingImsang, \
check_model_validity, write_log, rasterize_feature, quick_check, plot_raster

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys, traceback
import os
from glob import glob
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy.optimize import minimize
import joblib
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm
import numpy as np
import random
from datetime import datetime

# Libraries for ML-based learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import xgboost as xgb
from xgboost import XGBRegressor
import optuna
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, GridSearchCV, KFold, GroupKFold
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestRegressor
import shap
from xgboost import plot_importance
import warnings
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from functools import partial
import joblib

# Libraries for the pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# Libraries for the spatial analysis 
import geopandas as gpd
import scipy.stats as stats
import rasterio
import fiona
from rasterio.windows import Window
from collections import defaultdict
from rasterio.transform import xy
from rasterio.windows import transform
from rasterio.features import rasterize
from rasterio.plot import show

try:
    import pyproj
    from pyproj import CRS
except ImportError as e:
    print(e)
    usr_site = site.getusersitepackages()
    if usr_site in sys.path:
        sys.path.remove(usr_site)   # stop picking up pip user packages
        print("Removed user site:", usr_site)
    
    # Now import safely
    import pyproj
    from pyproj import CRS
    print("pyproj OK") 
    
# Libraries for the GPU use
import cupy as cp

# warning filter
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

In [ ]:
result_dir = r"D:/ForestFire/CBH/result/Baseline3"
fig_dir = r"D:\ForestFire\CBH\fig"
data_file = r"NFI6+7_train_combined_HM변형-try1.0.csv"
df = pd.read_csv(os.path.join(result_dir, data_file), encoding='cp949')

# encoded species code
le = LabelEncoder()
df['SID_ENC'] = le.fit_transform(df['SID'])

# Prepare X, y, species
numeric_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'Lat', 'Long', 'CR_pred']
X = df[feature_cols]
y = df[target_col]
species = df[cat_col].astype(str)
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# read test file
test_file = r"NFI6+7_test_combined_HM변형-try1.0.csv"
df_test = pd.read_csv(os.path.join(result_dir, test_file), encoding='cp949')

# label encoding
df_test['SID_ENC'] = le.transform(df_test['SID'])

feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
cat_col = 'SID'  # species id
target_col = 'CR'  # assuming target column name is 'CR'

# X, Y
X_test = df_test[feature_cols]
y_test = df_test[target_col]

In [ ]:
# encoded species code
le = LabelEncoder()
df['SID_ENC'] = le.fit_transform(df['SID'])
len(le.classes_)

In [ ]:
model = joblib.load(os.path.join(model_dir, "HM변형-try1.0-XGBoostGlobal.pkl"))

In [ ]:
# Font
plt.rcParams['font.family'] = 'Calibri'     # Set Calibri as the global font
plt.rcParams['font.size'] = 11              # Optional: slightly smaller default font

# Feature importance graph
feature_names = ['Height(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'Species', 'Latitude', 'Longitude', f'Allometric\nPrediction']

# Get feature importances
importance = model.feature_importances_

# Sort features by XGBoost importance (for consistent display)
sorted_idx = np.argsort(importance)
sorted_features = [feature_names[i] for i in sorted_idx]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

bar_height = 0.5
y = np.arange(len(feature_names))

bars = ax.barh(y - bar_height/2, importance[sorted_idx], height=bar_height, 
               label='Feature Importance', color='gray', alpha=0.6)

# Add value labels
for bar in bars:
    width = bar.get_width()
    y_pos = bar.get_y() + bar.get_height() / 2
    ax.text(width + 0.005, y_pos, f"{width:.3f}", va="center", ha="left", fontsize=12, weight='bold')

# Final touches
ax.set_yticks(x - 0.2)
ax.set_yticklabels(sorted_features, rotation=45, ha='right', fontsize=15, weight='bold')
ax.set_xlabel("Mean Feature Importance (Gain)", fontsize=15, weight='bold', labelpad=12)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=15, weight='bold')
# ax.set_title("Feature Importance", fontsize=24)
ax.legend(fontsize=15, borderaxespad=1.25, frameon=True)

# Extend the x-axis limit to the right for better spacing
xmax = max(importance) * 1.1
ax.set_xlim(0, xmax)

# Add gridlines in the background
ax.grid(axis='x', linestyle='--', color='lightgray', alpha=0.5)
ax.set_axisbelow(True)  # ensure gridlines are behind the bars

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "FeatureImportance1.0.png"), dpi=300)
plt.show()


In [ ]:
# ==== Visualize: SHAP value ====
X = df[feature_cols]
explainer = shap.Explainer(model)
shap_values = explainer(X)

In [ ]:
plt.rcParams['font.size'] = 15 
fig, ax = plt.subplots(figsize=(15, 8))
shap.summary_plot(shap_values, X, show=False, plot_size=(15, 8))

ax = plt.gca()
fig = plt.gcf()

# Add vertical gridlines (behind points)
ax.grid(axis='x', linestyle='-', color='gray', alpha=0.4)
ax.set_axisbelow(True)

# Replace y-tick labels with more human-friendly names
pretty_names = {
    'CR_pred': f'Allometric\nPrediction',
    'Lat': 'Latitude',
    'SID_ENC': 'Species',
    'Elev(hm)': 'Elevation (hm)',
    'Long': 'Longitude',
    'H(ft)': 'Height (ft)',
    'CD(%)': 'Crown Density (%)',
    'Slope(tan)': 'Slope (tan)',
    'DBH(inch)': 'DBH (inch)',
    'Azimuth(rad)': 'Azimuth (rad)'
}

yticklabels = [pretty_names.get(label.get_text(), label.get_text())
               for label in ax.get_yticklabels()]
ax.set_yticklabels(yticklabels, fontsize=20, weight='bold')
ax.set_xticklabels(ax.get_xticklabels(),fontsize=18, weight='bold')
# Label and title
ax.set_xlabel("SHAP value", fontsize=23, labelpad=18, weight='bold')
ax.set_ylabel("")  # optional: hide redundant label
ax.set_ylim(-1, len(yticklabels) - 1 + 0.8)

# Tweak layout
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "SHAP1.0.png"), dpi=300)
plt.show()

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

pdp_features = ['SID_ENC'] # 'CR_pred', 'Lat', 'Elev(hm)', 'Long', 'H(ft)'

# Optional: global style
plt.rcParams['font.family'] = 'Calibri'
plt.rcParams['axes.labelweight'] = 'normal'
plt.rcParams['axes.titleweight']  = 'semibold'

pretty = {
    'Long':'Longitude', 'Lat':'Latitude', 'Elev(hm)':'Elevation (hm)', 'H(ft)':'Height (ft)',
    'DBH(inch)':'DBH (inch)', 'CD(%)':'Crown Density (%)', 'Slope(tan)':'Slope (tan)',
    'Azimuth(rad)':'Azimuth (rad)', 'CR_pred':'Allometric Prediction on Crown Ratio',
    'SID_ENC' : 'Species (encoded)'
}

for feat in pdp_features:                      # e.g., list[str] of 1D features
    fig, ax = plt.subplots(figsize=(10, 7))
    disp = PartialDependenceDisplay.from_estimator(
        model, X, [feat],
        kind='average',
        grid_resolution=100,
        ax=ax,                          # ← remove the vertical rug ticks
        pd_line_kw={'color': 'red', 'lw': 3.0}   # ← set PDP line color here
    )
    
    # use the axis actually used by the display (safe even if sklearn makes sub-axes)
    ax = disp.axes_.ravel()[0]
    
# ===== Remove decile/rug spikes (robust across sklearn versions) =====
    # A) If drawn as LineCollections
    for coll in list(ax.collections):
        if isinstance(coll, LineCollection):
            coll.remove()

    # B) If drawn as short vertical Line2D segments
    yspan = np.diff(ax.get_ylim())[0]
    for ln in list(ax.lines)[1:]:  # keep the first line (PDP curve)
        x = ln.get_xdata()
        y = ln.get_ydata()
        is_vertical = np.allclose(x, x[0])      # x is constant
        is_short = (np.ptp(y) < 0.15 * yspan)   # tiny y-extent
        if is_vertical and is_short:
            ln.remove()
            
    # style *after* drawing
    # ax.set_facecolor('#FAFAFA')
    ax.grid(True, linestyle='--', color='0.85', linewidth=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # labels/titles
    ax.set_xlabel(pretty.get(feat, feat), fontsize=15, weight='bold', labelpad=6)
    ax.set_ylabel("Partial dependence of predicted CBH", fontsize=15, weight='bold')
    # ax.set_title(f"PDP — {pretty.get(feat, feat)}", fontsize=16, pad=10)
    
    # tighten y-limits to the curve so the plot doesn’t look empty
    line = ax.lines[0]
    ymin, ymax = np.min(line.get_ydata()), np.max(line.get_ydata())
    pad = 0.1 * (ymax - ymin + 1e-12)
    ax.set_ylim(ymin - pad, ymax + pad)
    
    fig.tight_layout()
    fig.savefig(os.path.join(fig_dir, f"pdp_{feat}.png"), dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
# If you want to replot onto a new layout...
fig2, axs2 = plt.subplots(nrows=2, ncols=3, figsize=(12, 7))
axs2 = axs2.ravel()
disp.plot(ax=axs2)          # reuses computed results; does NOT recompute
fig2.tight_layout()
fig2.savefig("pdp_cbh_alt.png", dpi=300, bbox_inches='tight')

In [ ]:
# violin plot
target_feat = 'DMCLS' # AGECLS
compare1_file = os.path.join(result_dir, "Pred-NIFS-ComparebyCategory_CR4.csv")
if os.path.exists(compare1_file):
    print('yes')

df = pd.read_csv(compare1_file, encoding='cp949')
df1 = df[[target_feat, 'mean', 'NIFS-CBH']].dropna()
df1 = df1.loc[df1['NIFS-CBH']!= '-']
# Ensure AGECLS is ordered (numeric or categorical order)
df1[target_feat] = df1[target_feat].astype('int')
df1['NIFS-CBH'] = df1['NIFS-CBH'].astype('float')
ages = sorted(df1[target_feat].unique())

In [ ]:
from matplotlib.patches import Patch 
pred_fill = "#A6A6A6"
pred_edge = 'black'
nifs_fill = '#DCEAF7'
nifs_edge = '#163E64'

# ---- Build data lists per age class ----
pred_data, nifs_data = [], []
for a in ages:
    sub = df1[df1[target_feat] == a]
    pred_data.append(sub['mean'].values)
    nifs_data.append(sub['NIFS-CBH'].values)

# ---- Plot grouped boxplots ----
plt.rcParams['font.family'] = 'Calibri'  # optional

fig, ax = plt.subplots(figsize=(9, 5))

n_groups = len(ages)
spacing = 3  # increase this for more spacing (try 2, 3, or 5)
centers = np.arange(1, n_groups * spacing + 1, spacing, dtype=float)

width = 1  # half-distance between the two boxes in each group
pos_pred = centers - width/2
pos_nifs = centers + width/2

# Draw the two sets
bp_pred = ax.boxplot(pred_data, positions=pos_pred, widths=0.7,
                     patch_artist=True, showmeans=True,
                     meanprops=dict(marker='x', markersize=6, markeredgecolor=pred_edge),
                     boxprops=dict(color=pred_edge, linewidth=1),
                     whiskerprops=dict(color=pred_edge, linewidth=1),
                     capprops=dict(color=pred_edge, linewidth=1),
                     medianprops=dict(color='black', linewidth=1),
                     flierprops=dict(
                         marker='o',
                         markersize=4,
                         markerfacecolor=pred_fill,  # or match box color
                         markeredgecolor=pred_edge,
                         alpha=0.5
                     ))

bp_nifs = ax.boxplot(nifs_data, positions=pos_nifs, widths=0.7,
                     patch_artist=True, showmeans=True,
                     meanprops=dict(marker='x', markersize=6, markeredgecolor=nifs_edge),
                     boxprops=dict(color=nifs_edge, linewidth=1),
                     whiskerprops=dict(color=nifs_edge, linewidth=1),
                     capprops=dict(linewidth=1),
                     medianprops=dict(color=nifs_edge, linewidth=1),
                     flierprops=dict(
                         marker='o',
                         markersize=4,
                         markerfacecolor=nifs_edge,  # or match box color
                         markeredgecolor=nifs_edge,
                         alpha=0.5
                     ))
    
# Colors
for patch in bp_pred['boxes']:
    patch.set_facecolor(pred_fill)   # Prediction (blue-ish)
for patch in bp_nifs['boxes']:
    patch.set_facecolor(nifs_fill)   # NIFS (green-ish)

# Axes & labels
ax.set_xticks(centers)
ax.set_xticklabels([str(a) for a in ages])
ax.set_xlabel("Tree DBH Class", fontsize=12, weight='bold') # "Tree Age Class"
ax.set_ylabel("Crown Base Height (m)", fontsize=12, weight='bold')
# ax.set_title("Crown Base Height by Tree Age Class (Prediction vs NIFS)", fontsize=14, weight='bold')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

padding = 1.5  # try 1.2 or 1.5 for more space
ax.set_xlim(centers[0] - padding, centers[-1] + padding)
plt.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.12)


# Legend (proxy patches)
legend_handles = [
    Patch(facecolor=pred_fill, edgecolor='black', label='Prediction (mean)'),
    Patch(facecolor=nifs_fill, edgecolor='black', label='CBH from NIFS')
]
ax.legend(handles=legend_handles, loc='upper left', frameon=True, fontsize=14)

fig.tight_layout()
plt.savefig(os.path.join(fig_dir, f"boxplot-NIFSvsPred-{target_feat}.png"), dpi=300)
plt.show()